# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. **Select a GPU runtime before running this notebook.** 

Original numbered course notebook, adapted for Colab. Run in numerical order: later lessons load models trained by earlier lessons. Use the same Drive result folder across sessions. W&B logs stay offline. Full training can exceed a free Colab session; QUICK_RUN does not shorten these original Part 2 experiments.

Source: https://github.com/fastai/course22p2/blob/df9323235bc395b5c2f58a3d08b83761947b9b93/nbs/30_lsun_diffusion-latents.ipynb

In [ ]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '04_fastai_part2', 'part2', '30_lsun_diffusion-latents.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

## Full exercise evidence

- Run all exercise cells top to bottom in hosted Colab and preserve their outputs, errors and execution counts.
- Keep QUICK_RUN=False; shortened checks or synthetic replacements for real datasets do not count as full original exercises.
- Preserve plots, models, metrics and run configurations in Drive; a final-cell marker alone is insufficient.
- Retain original datasets, model definitions, training schedules and instructional comparisons.
- Run prerequisite checkpoint-producing notebooks first; do not replace missing weights with untrained models.

- Before training, verify all303,125 original LSUN JPEGs against the exact course archive and all303,125 finite float32(4,32,32) latent rows. Resume missing/corrupt rows from a per-row SHA256 ledger; file allocation/existence is insufficient.

This cell describes evidence to collect; it does not certify completion. GPU batch caps preserve dataset coverage and epoch counts, but are documented configuration changes.

In [ ]:
MAX_BATCH_SIZE = 8  #@param {type:"integer"}
# Increase on a larger GPU to match the original batch sizes.
assert MAX_BATCH_SIZE > 0

# LSUN Bedrooms

In [ ]:
import os
# Colab provides the visible GPU; keep its device selection.

In [ ]:
from miniai.imports import *
from miniai.diffusion import *

from glob import glob
from fastprogress import progress_bar
from diffusers import AutoencoderKL, UNet2DConditionModel

In [ ]:
import timm

In [ ]:
torch.set_printoptions(precision=4, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['image.cmap'] = 'gray_r'
mpl.rcParams['figure.dpi'] = 70

set_seed(42)
if fc.defaults.cpus>2: fc.defaults.cpus=2

## Data processing

In [ ]:
path_data = Path('data')
path_data.mkdir(exist_ok=True)
path = path_data/'bedroom'

In [ ]:
"""Inlined into notebooks30/31; functions only, with no downloads or training on import."""
import csv, gzip, hashlib, json, os, shutil, sqlite3, tarfile, time
from pathlib import Path
import numpy as np

def cache_sha_file(filename):
    value = hashlib.sha256()
    with open(filename, 'rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            value.update(block)
    return value.hexdigest()

def cache_json_sha(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',',':'), default=str).encode()).hexdigest()

def cache_atomic_json(filename, value):
    filename = Path(filename); filename.parent.mkdir(parents=True, exist_ok=True)
    temporary = filename.with_name(filename.name+'.writing')
    with temporary.open('w', encoding='utf-8') as stream:
        json.dump(value, stream, sort_keys=True, indent=2); stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, filename)

def latent_npy_disk_plan(cache_root, rows):
    """Account for NPY headers, filesystem blocks and available file entries."""
    import io
    cache_root = Path(cache_root)
    sample = io.BytesIO(); np.save(sample,np.empty((4,32,32),np.float32),allow_pickle=False)
    logical_row_bytes = sample.tell()
    fs = os.statvfs(cache_root) if hasattr(os,'statvfs') else None
    block = (fs.f_frsize or fs.f_bsize) if fs else 512
    allocated_row_bytes = ((logical_row_bytes+block-1)//block)*block
    existing = list(cache_root.rglob('*.npy'))
    allocated = sum(min(getattr(p.stat(),'st_blocks',0)*512 or p.stat().st_size,allocated_row_bytes) for p in existing)
    required_free = max(rows*allocated_row_bytes-allocated,0)+1024**3
    if shutil.disk_usage(cache_root).free < required_free:
        raise OSError(f'Full original float32 NPY cache needs {required_free:,} additional runtime bytes, including file-block overhead and manifest reserve.')
    if fs and fs.f_files and fs.f_favail < max(rows-len(existing),0)+2100:
        raise OSError('Runtime filesystem has insufficient free file entries for the complete original NPY cache.')
    return {'npy_bytes_per_row':logical_row_bytes,'filesystem_block_bytes':block,
            'allocated_bytes_per_row':allocated_row_bytes,'full_allocated_bytes':rows*allocated_row_bytes}

def full_lsun_source(path, url, evidence, expected_images=303125):
    """Verify every original tar member; repair missing/corrupt JPEGs atomically."""
    import urllib.request
    path = Path(path); evidence = Path(evidence); evidence.mkdir(parents=True, exist_ok=True)
    archive = path.parent/'bedroom.tgz'
    if not archive.is_file():
        request = urllib.request.Request(url, method='HEAD')
        with urllib.request.urlopen(request, timeout=60) as response:
            advertised_bytes = int(response.headers.get('Content-Length',0))
        free = shutil.disk_usage(path.parent).free
        if advertised_bytes and free < advertised_bytes + 1024**3:
            raise OSError('Insufficient runtime disk for original LSUN archive; data are not stored in Drive.')
        partial = archive.with_name(archive.name+'.downloading')
        # No partial HTTP download is treated as a complete archive.
        with urllib.request.urlopen(url, timeout=120) as response, partial.open('wb') as output:
            shutil.copyfileobj(response, output, 1024*1024); output.flush(); os.fsync(output.fileno())
        if advertised_bytes and partial.stat().st_size != advertised_bytes:
            raise IOError('LSUN archive download is truncated.')
        os.replace(partial, archive)
    source_hashes = {}; aggregate = hashlib.sha256(); extracted_bytes = 0
    source_record = evidence/'original_lsun_source.jsonl.gz'
    temporary_record = source_record.with_name(source_record.name+'.writing')
    try:
        with tarfile.open(archive, 'r:gz') as source, gzip.open(temporary_record, 'wt', encoding='utf-8') as manifest:
            members = [m for m in source.getmembers() if m.isfile() and m.name.lower().endswith('.jpg')]
            if len(members) != expected_images:
                raise ValueError(f'Original LSUN archive must contain {expected_images} JPEGs; found {len(members)}.')
            expected_targets = set()
            missing_bytes = 0
            for member in members:
                parts = Path(member.name).parts
                if not parts or parts[0] != 'bedroom' or '..' in parts or Path(member.name).is_absolute():
                    raise ValueError('Unsafe or unexpected original LSUN archive path.')
                target = (path.parent/member.name).resolve()
                if not target.is_relative_to(path.resolve()) or target in expected_targets:
                    raise ValueError('Unsafe or duplicate original LSUN JPEG path.')
                expected_targets.add(target); extracted_bytes += member.size
                if not target.is_file() or target.stat().st_size != member.size:
                    missing_bytes += member.size
            if shutil.disk_usage(path.parent).free < missing_bytes + 1024**3:
                raise OSError('Insufficient runtime disk for extracted LSUN images; use a larger runtime disk.')
            for member in members:
                target = (path.parent/member.name).resolve(); target.parent.mkdir(parents=True, exist_ok=True)
                # Hash the authoritative JPEG bytes inside the course's exact archive.
                with source.extractfile(member) as input_stream:
                    original_bytes = input_stream.read()
                if len(original_bytes) != member.size: raise IOError('Truncated JPEG inside original LSUN archive.')
                source_sha = hashlib.sha256(original_bytes).hexdigest()
                if not target.is_file() or target.stat().st_size != member.size or cache_sha_file(target) != source_sha:
                    temporary = target.with_name(target.name+'.restoring')
                    with temporary.open('wb') as output:
                        output.write(original_bytes); output.flush(); os.fsync(output.fileno())
                    if temporary.stat().st_size != member.size or cache_sha_file(temporary) != source_sha:
                        raise IOError('LSUN JPEG restoration failed: '+member.name)
                    os.replace(temporary, target)
                relative = target.relative_to(path.resolve()).as_posix()
                source_hashes[relative] = source_sha
                line = json.dumps({'relative_path':relative,'bytes':member.size,'sha256':source_sha}, separators=(',',':'))
                aggregate.update((line+'\n').encode()); manifest.write(line+'\n')
            actual = {p.resolve() for p in path.rglob('*.jpg')}
            if actual != expected_targets:
                raise ValueError('LSUN directory differs from the exact archive JPEG inventory; no extra/omitted images are allowed.')
        os.replace(temporary_record, source_record)
        summary = {'original_url':url,'archive_bytes':archive.stat().st_size,'archive_sha256':cache_sha_file(archive),
                   'extracted_jpeg_bytes':extracted_bytes,'expected_images':expected_images,
                   'verified_images':len(source_hashes),'source_manifest_sha256':aggregate.hexdigest(),
                   'data_location':'runtime-local','course_completion_claimed':False}
        cache_atomic_json(evidence/'original_lsun_source_status.json', summary)
        print('Verified exact LSUN source:',summary)
        return source_hashes
    except BaseException as error:
        cache_atomic_json(evidence/'original_lsun_source_status.json', {'source_complete':False,'error':str(error),'course_completion_claimed':False})
        raise

def full_imagenet_source(path, path_data, files, expected_train=1281167, expected_val=50000, expected_classes=1000):
    """Require exact full splits and resolve official labels, including flat val JPEGs."""
    import xml.etree.ElementTree as ET
    path = Path(path); path_data = Path(path_data)
    id_file = path_data/'imagenet_lsvrc_2015_synsets.txt'; words_file = path_data/'words.txt'
    if not id_file.is_file() or not words_file.is_file():
        raise FileNotFoundError('Supply original ImageNet synset-order and words files under data/ILSVRC; no Tiny ImageNet replacement is used.')
    ids = id_file.read_text().splitlines(); id_set = set(ids)
    if len(ids) != expected_classes or len(id_set) != expected_classes or any(not s.startswith('n') or not s[1:].isdigit() for s in ids):
        raise ValueError('ImageNet synset-order file must contain exactly1000 distinct original synsets.')
    words = dict(line.split('\t',1) for line in words_file.read_text().splitlines() if '\t' in line)
    if not id_set.issubset(words): raise ValueError('ImageNet words file does not cover all required synsets.')
    val_csv = next((p for p in [path_data/'LOC_val_solution.csv',path_data.parent/'LOC_val_solution.csv'] if p.is_file()), None)
    csv_labels = {}
    if val_csv:
        with val_csv.open(newline='') as stream:
            for row in csv.DictReader(stream):
                labels = {word for word in row['PredictionString'].split() if word in id_set}
                if len(labels) != 1: raise ValueError('Official validation row has ambiguous classification synset: '+row['ImageId'])
                csv_labels[row['ImageId']] = next(iter(labels))
    targets = []; counts = {'train':0,'val':0,'test':0}; class_counts = {key:0 for key in ids}; val_class_counts = {key:0 for key in ids}; label_inputs = [id_file,words_file]
    if val_csv: label_inputs.append(val_csv)
    annotation_hash = hashlib.sha256(); seen = set()
    for filename in files:
        filename = Path(filename); relative = filename.resolve().relative_to(path.resolve())
        split = relative.parts[0]
        if split not in counts: raise ValueError('Unexpected CLS-LOC split: '+str(relative))
        counts[split] += 1
        if split == 'train':
            label = relative.parts[1]
            if label not in id_set: raise ValueError('Unknown training synset: '+label)
            class_counts[label] += 1
        elif split == 'val':
            label = filename.parent.name if filename.parent.name in id_set else csv_labels.get(filename.stem)
            if label is None:
                annotation = path_data/'Annotations'/'CLS-LOC'/'val'/(filename.stem+'.xml')
                if not annotation.is_file(): raise FileNotFoundError('Flat ImageNet validation JPEGs require their official val XML or LOC_val_solution.csv: '+filename.name)
                labels = {node.text for node in ET.parse(annotation).findall('./object/name')}
                if len(labels) != 1 or not labels.issubset(id_set): raise ValueError('Ambiguous original validation annotation: '+annotation.name)
                label = next(iter(labels)); annotation_hash.update((annotation.name+cache_sha_file(annotation)).encode())
            val_class_counts[label] += 1
        else: label = ''
        target = Path(split)/(label or 'unlabelled')/(filename.stem+'.npy')
        if target.as_posix() in seen: raise ValueError('Duplicate latent target: '+str(target))
        seen.add(target.as_posix()); targets.append(target.as_posix())
    if counts['train'] != expected_train or counts['val'] != expected_val or not all(class_counts.values()):
        raise ValueError(f'Full ImageNet requires train={expected_train}, val={expected_val}, all{expected_classes}classes; observed {counts}.')
    if expected_val % expected_classes or any(count != expected_val//expected_classes for count in val_class_counts.values()):
        raise ValueError('Original ImageNet validation must include exactly50 images for each of the1000 synsets.')
    if counts['test'] not in [0,100000]: raise ValueError('If supplied, the optional unlabelled test split must be complete.')
    labels_fingerprint = cache_json_sha({'files':{p.name:cache_sha_file(p) for p in label_inputs},'xml_labels_sha256':annotation_hash.hexdigest(),'synset_order':ids})
    return targets, {'counts':counts,'class_counts':class_counts,'validation_class_counts':val_class_counts,'synset_order_sha256':cache_sha_file(id_file),'label_sources_sha256':labels_fingerprint}

def latent_encoder_settings(encoder, preprocessing):
    import torch, torchvision
    weights = hashlib.sha256()
    for name, tensor in sorted(encoder.state_dict().items()):
        array = tensor.detach().cpu().contiguous().numpy()
        weights.update(name.encode()); weights.update(str(array.shape).encode()); weights.update(str(array.dtype).encode()); weights.update(array.tobytes())
    return {'model_id':'stabilityai/sd-vae-ft-ema','weights_sha256':weights.hexdigest(),
            'config':dict(encoder.config),'preprocessing':preprocessing,
            'latent_statistic':'latent_dist.mean','shape':[4,32,32],'dtype':'float32',
            'torch':torch.__version__,'torchvision':torchvision.__version__}

def verified_full_latents(files, source_root, get_image, encoder, cache_base, evidence, settings,
                          mode='npy', targets=None, batch_size=4, source_proof=None, snapshot_seconds=300):
    """Every row must have current source and output hashes; interrupted rows resume."""
    import torch
    files = [Path(p) for p in files]; source_root = Path(source_root).resolve()
    relatives = [p.resolve().relative_to(source_root).as_posix() for p in files]
    if len(set(relatives)) != len(files) or not files: raise ValueError('Source inventory is empty or duplicated.')
    if mode not in ['npy','memmap']: raise ValueError(mode)
    if targets is None: targets = [str(i) for i in range(len(files))]
    if len(targets) != len(files) or len(set(targets)) != len(files): raise ValueError('Latent row manifest is not one-to-one.')
    order = hashlib.sha256()
    for relative,target in zip(relatives,targets): order.update((relative+'\t'+target+'\n').encode())
    identity = {'settings':settings,'source_order_sha256':order.hexdigest(),'rows':len(files),'mode':mode}
    identity_sha = cache_json_sha(identity)
    cache_root = Path(cache_base)/('verified-'+identity_sha[:24]); cache_root.mkdir(parents=True, exist_ok=True)
    evidence = Path(evidence); evidence.mkdir(parents=True, exist_ok=True)
    snapshot = evidence/(identity_sha[:24]+'-manifest.sqlite3.gz')
    db_file = cache_root/'row-manifest.sqlite3'
    if not db_file.exists() and snapshot.exists():
        temporary = db_file.with_name(db_file.name+'.restoring')
        with gzip.open(snapshot,'rb') as source, temporary.open('wb') as output: shutil.copyfileobj(source,output)
        os.replace(temporary,db_file)
    conn = sqlite3.connect(db_file)
    conn.execute('PRAGMA synchronous=FULL')
    if conn.execute('PRAGMA integrity_check').fetchone()[0] != 'ok': raise IOError('Corrupt latent row manifest; never treat it as completion.')
    conn.execute('CREATE TABLE IF NOT EXISTS metadata (key TEXT PRIMARY KEY,value TEXT)')
    conn.execute('CREATE TABLE IF NOT EXISTS rows (seq INTEGER PRIMARY KEY,relative TEXT,source_sha TEXT,latent_sha TEXT,complete INTEGER)')
    old = conn.execute("SELECT value FROM metadata WHERE key='identity'").fetchone()
    if old and old[0] != identity_sha: raise ValueError('Latent manifest encoding/source-order identity does not match.')
    conn.execute("INSERT OR REPLACE INTO metadata VALUES ('identity',?)",(identity_sha,)); conn.commit()
    output_file = cache_root/'data.npmm'; array = None
    shape = (len(files),4,32,32); total_bytes = int(np.prod(shape,dtype=np.int64))*np.dtype(np.float32).itemsize
    if mode == 'memmap':
        if output_file.exists() and output_file.stat().st_size != total_bytes:
            os.replace(output_file,output_file.with_name('invalid-size-'+str(time.time_ns())+'.npmm'))
            conn.execute('UPDATE rows SET complete=0');conn.commit()
        if not output_file.exists():
            if shutil.disk_usage(cache_root).free < total_bytes+1024**3: raise OSError('Insufficient runtime disk for full float32 latent memmap.')
            conn.execute('UPDATE rows SET complete=0');conn.commit()
            allocation = output_file.with_name(output_file.name+'.allocating')
            fresh = np.memmap(allocation,np.float32,mode='w+',shape=shape);fresh.flush();del fresh
            os.replace(allocation,output_file)
        array = np.memmap(output_file,np.float32,mode='r+',shape=shape)
    else: latent_npy_disk_plan(cache_root,len(files))
    started = time.monotonic(); last_snapshot = started; verified = 0; regenerated = 0; aggregate = hashlib.sha256(); ready = False
    status_file = evidence/(identity_sha[:24]+'-status.json')
    def checkpoint(error=None):
        nonlocal last_snapshot
        conn.commit(); copy_file = cache_root/'manifest-snapshot.sqlite3'
        copy = sqlite3.connect(copy_file)
        try: conn.backup(copy)
        finally: copy.close()
        temporary = snapshot.with_name(snapshot.name+'.writing')
        with copy_file.open('rb') as source, gzip.open(temporary,'wb',compresslevel=1) as output: shutil.copyfileobj(source,output,1024*1024)
        os.replace(temporary,snapshot);copy_file.unlink()
        cache_atomic_json(status_file,{'identity':identity,'identity_sha256':identity_sha,'expected_rows':len(files),
            'verified_current_source_rows':verified,'regenerated_this_attempt':regenerated,'cache_complete':ready,
            'source_content_manifest_sha256':aggregate.hexdigest(),'duration_seconds':time.monotonic()-started,
            'runtime_cache':str(cache_root),'latent_payload_bytes':total_bytes,'error':error,
            'data_location':'runtime-local; only compact manifest/status persisted','course_completion_claimed':False})
        last_snapshot = time.monotonic()
    def valid_latent(value):
        return value.shape == (4,32,32) and value.dtype == np.float32 and bool(np.isfinite(value).all())
    def read_latent(index):
        return np.asarray(array[index]) if mode == 'memmap' else np.load(cache_root/targets[index],allow_pickle=False)
    def process(todo):
        nonlocal regenerated
        if not todo: return
        batch = torch.stack([get_image(index) for index,sha in todo])
        device = next(encoder.parameters()).device
        with torch.inference_mode(): values = encoder.encode(batch.to(device)).latent_dist.mean.detach().cpu().numpy()
        if values.shape != (len(todo),4,32,32) or values.dtype != np.float32 or not np.isfinite(values).all():
            raise ValueError('Encoder did not produce complete finite original float32 mean latents.')
        rows = []
        for (index,sha),value in zip(todo,values):
            if cache_sha_file(files[index]) != sha: raise IOError('Source image changed during latent encoding: '+relatives[index])
            if mode == 'memmap': array[index] = value
            else:
                output = cache_root/targets[index];output.parent.mkdir(parents=True,exist_ok=True)
                if not output.resolve().is_relative_to(cache_root.resolve()): raise ValueError('Unsafe latent output path.')
                temporary = output.with_name(output.name+'.writing')
                with temporary.open('wb') as stream:
                    np.save(stream,value,allow_pickle=False);stream.flush();os.fsync(stream.fileno())
                os.replace(temporary,output)
            rows.append((index,relatives[index],sha,hashlib.sha256(value.tobytes()).hexdigest(),1))
        if mode == 'memmap':
            array.flush()
            with output_file.open('r+b') as stream: os.fsync(stream.fileno())
        conn.executemany('INSERT OR REPLACE INTO rows VALUES (?,?,?,?,?)',rows);conn.commit();regenerated += len(rows)
    try:
        checkpoint(); todo = []
        for index,filename in enumerate(files):
            # Re-read current source hashes on every attempt; metadata-only cache hits are insufficient.
            sha = cache_sha_file(filename)
            if source_proof is not None and source_proof.get(relatives[index]) != sha:
                raise ValueError('Current LSUN image differs from the verified original archive: '+relatives[index])
            aggregate.update((relatives[index]+'\t'+sha+'\t'+targets[index]+'\n').encode())
            row = conn.execute('SELECT relative,source_sha,latent_sha,complete FROM rows WHERE seq=?',(index,)).fetchone()
            valid = False
            if row and row[0] == relatives[index] and row[1] == sha and row[3]:
                try:
                    value = read_latent(index)
                    valid = valid_latent(value) and hashlib.sha256(value.tobytes()).hexdigest() == row[2]
                except (OSError,ValueError,EOFError): pass
            if not valid: todo.append((index,sha))
            if len(todo) >= batch_size: process(todo);todo=[]
            verified = index+1
            if time.monotonic()-last_snapshot >= snapshot_seconds:
                process(todo);todo=[];checkpoint()
        process(todo)
        if conn.execute('SELECT COUNT(*) FROM rows WHERE complete=1').fetchone()[0] != len(files):
            raise ValueError('Latent row manifest is incomplete; training is blocked.')
        if mode == 'npy':
            actual = {p.relative_to(cache_root).as_posix() for p in cache_root.rglob('*.npy')}
            if actual != set(targets): raise ValueError('Latent file inventory has extra or missing rows.')
        ready = True;checkpoint()
        print(f'Verified {len(files):,} current-source latent rows; regenerated {regenerated:,}; data remain runtime-local.')
        return {'cache_root':cache_root,'memmap':output_file if mode=='memmap' else None,
                'files':[str(cache_root/target) for target in targets] if mode=='npy' else [],
                'identity_sha256':identity_sha,'source_manifest_sha256':aggregate.hexdigest(),
                'verified_rows':len(files),'regenerated_rows':regenerated,'cache_complete':True}
    except BaseException as error:
        ready = False;checkpoint(str(error));raise
    finally:
        if array is not None: array.flush();del array
        conn.close()

In [ ]:
url = 'https://s3.amazonaws.com/fast-ai-imageclas/bedroom.tgz'
lsun_source_hashes = full_lsun_source(path,url,Path('results')/'lsun_original_source')
assert len(lsun_source_hashes) == 303125

In [ ]:
bs = min(64, MAX_BATCH_SIZE)

In [ ]:
def to_img(f): return read_image(f, mode=ImageReadMode.RGB)/255

In [ ]:
class ImagesDS:
    def __init__(self, spec):
        self.path = Path(path)
        self.files = glob(str(spec), recursive=True)
    def __len__(self): return len(self.files)
    def __getitem__(self, i): return to_img(self.files[i])[:, :256,:256]

In [ ]:
ds = ImagesDS(path/f'**/*.jpg')
assert len(ds) == 303125, 'The complete original course LSUN subset is required.'
assert {Path(f).resolve().relative_to(path.resolve()).as_posix() for f in ds.files} == set(lsun_source_hashes)

In [ ]:
dl = DataLoader(ds, batch_size=min(4,bs), num_workers=fc.defaults.cpus)
xb = next(iter(dl))
show_images(xb[:16], imsize=2)

In [ ]:
xb[:16].shape

In [ ]:
16*3*256*256

## VAE

In [ ]:
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-ema").cuda().requires_grad_(False)

In [ ]:
xe = vae.encode(xb.cuda())

In [ ]:
xs = xe.latent_dist.mean[:16]
xs.shape

In [ ]:
(16*3*256*256)/(16*4*32*32)

In [ ]:
show_images(((xs[:16,:3])/4).sigmoid(), imsize=2)

In [ ]:
xd = to_cpu(vae.decode(xs))
show_images(xd['sample'].clamp(0,1), imsize=2)

In [ ]:
mmpath = Path('data/bedroom/data.npmm')

In [ ]:
len(ds)

In [ ]:
mmshape = (303125,4,32,32)

In [ ]:
# Existing allocation is not evidence that every source image was encoded.
lsun_encoding = latent_encoder_settings(vae,{'rgb':'read_image RGB /255','crop':'top-left256x256','resize':None})
lsun_latents = verified_full_latents(
    ds.files,path,lambda index: ds[index],vae,Path('data/bedroom/latent-cache'),
    Path('results')/'lsun_latent_preparation',lsun_encoding,mode='memmap',
    batch_size=min(4,bs),source_proof=lsun_source_hashes)
assert lsun_latents['cache_complete'] and lsun_latents['verified_rows'] == 303125
mmpath = lsun_latents['memmap']

In [ ]:
lats = np.memmap(mmpath, dtype=np.float32, mode='r', shape=mmshape)

In [ ]:
b = torch.tensor(lats[:16])

In [ ]:
xd = to_cpu(vae.decode(b.cuda()))
show_images(xd['sample'].clamp(0,1), imsize=2)

## Noisify

In [ ]:
def collate_ddpm(b): return noisify(default_collate(b)*0.2)

In [ ]:
n = len(lats)

In [ ]:
tds = lats[:n//10*9 ]
vds = lats[ n//10*9:]

In [ ]:
bs = min(128, MAX_BATCH_SIZE)

In [ ]:
dls = DataLoaders(*get_dls(tds, vds, bs=bs, num_workers=fc.defaults.cpus, collate_fn=collate_ddpm))

In [ ]:
import warnings

In [ ]:
warnings.simplefilter('ignore', UserWarning)

In [ ]:
(xt,t),eps = b = next(iter(dls.train))

In [ ]:
show_images(xt[:9,0], imsize=1.5)

In [ ]:
xte = vae.decode(xt[:9].cuda()*5)['sample']
show_images(xte.clamp(0,1), imsize=1.5)

## Train

In [ ]:
def init_ddpm(model):
    for o in model.downs:
        for p in o.resnets: p.conv2[-1].weight.data.zero_()

    for o in model.ups:
        for p in o.resnets: p.conv2[-1].weight.data.zero_()

In [ ]:
lr = 3e-3
epochs = 25
opt_func = partial(optim.AdamW, eps=1e-5)
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
cbs = [DeviceCB(), ProgressCB(plot=True), MetricsCB(), BatchSchedCB(sched), MixedPrecision()]
model = EmbUNetModel(in_channels=4, out_channels=4, nfs=(128, 256, 512, 768), num_layers=2,
                     attn_start=1, attn_chans=16)
init_ddpm(model)
learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)

In [ ]:
learn.fit(epochs)

In [ ]:
# torch.save(learn.model.state_dict(), 'models/lsun_diffusion-latents.pkl')

## Sampling

In [ ]:
sz = (16,4,32,32)

In [ ]:
# set_seed(42)
preds = sample(ddim_step, model, sz, steps=100, eta=1., clamp=False)

In [ ]:
s = preds[-1]*5

In [ ]:
with torch.no_grad(): pd = to_cpu(vae.decode(s.cuda()))

In [ ]:
show_images(pd['sample'][:9].clamp(0,1), imsize=5)

Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [ ]:
finish_run(COLAB_CONTEXT)

## Original LSUN source and cache completeness

The course archive is verified JPEG by JPEG and partial extraction is repaired.
All303,125 images are required; the original file traversal and teaching split
are retained. Archive, extracted JPEGs and any full float32 latent cache remain
on the runtime disk. Only compact source/row manifests, status and existing
model/results artifacts persist in Drive. A new runtime may need the same
archive and latent regeneration; a manifest alone contains no image/latent data.

The full memmap (latent variant) is4,966,400,000 bytes. Raw data/archive and package
caches need additional local disk; the code checks available space. Do not put
the full cache into the account's roughly10GB free Drive. Every latent row is
bound to its source JPEG, preprocessing, exact VAE weights and float32 shape;
an allocated file or interrupted generation is never treated as complete.
The original25-epoch latent/15-epoch image model schedules remain.

The uploader's documented serialization of the same303,125 images is4.76GB;
that is a size reference, not a measured size of the course tar archive.
[public source description](https://huggingface.co/datasets/pcuenq/lsun-bedrooms).
No hosted execution or complete course status is claimed by these repairs.